# Databridge: load → verify → convert → consume

*A runnable tour of the bridge architecture, on synthetic data.*

Machine-learning teams receive object-detection / tracking datasets in many
incompatible on-disk formats and constantly convert between them. **databridge**
turns each input format into one neutral in-memory model, so any input format can
be re-exported to any output format without writing a bespoke converter for every
pair.

This notebook builds a tiny **synthetic** dataset, loads it, validates it,
converts it to MOTChallenge on disk, and consumes it as a MAITE
multi-object-tracking dataset — all on generic placeholder data, so it runs in a
clean checkout and is safe to commit (no real dataset content is used).

**The shape of the tool**

```
  loaders                                       neutral model           writers
  (_formats/<fmt>/loader.py)                     (model.py)             (_formats/<fmt>/writer.py)

  load_mot(..., dataset_format="hmie")         ─┐                  ┌─► write(..., output_format="hmie")
  load_mot(..., dataset_format="motchallenge") ─┼─► BoxTrackDataset ┼─► write(..., output_format="motchallenge")
  load_mot(..., dataset_format="tao")          ─┤   VideoSequence   ├─► write(..., output_format="tao")
  load_mot(..., dataset_format="visdrone")     ─┘   BoxAnnotation   └─► write(..., output_format="visdrone")
```

Every MOT loader produces the **same** `BoxTrackDataset`, and every MOT writer
consumes `BoxTrackDataset` — so any box-track/MOT input can feed any
box-track/MOT output (an N-to-M bridge) rather than a one-off HMIE→X path.
Task siblings such as video classification use their own in-memory models and
writer surfaces. `write()` exports the loaded dataset to disk
(section 3; `convert()` is the disk-to-disk shorthand pairing a load with a
write), and the loaded dataset itself natively implements the MAITE
multi-object-tracking protocol, so MAITE tooling consumes it without an adapter
(section 4).

**Terms used here**

- **HMIE / Scale** — the input dataset this loader reads. *Scale* is the
  annotation JSON schema (Scale AI's Video Playback format); *HMIE* is the
  on-disk folder layout it arrives in.
- **AFR** (annotation frame rate) — how many frames per second were labeled,
  which is often lower than the video's native FPS.
- **FMV** — full-motion video (the source clips).
- **MOTChallenge / TAO / VisDrone** — common detection/tracking dataset formats;
  the implemented export targets.
- **MAITE** — the JATIC protocol library for ML test & evaluation. Its dataset
  protocols are structural, and `BoxTrackDataset` conforms by shape.

## Prerequisites

This notebook imports the installed `databridge` package, so install it first
from a clone of the repo, then select that environment as the notebook kernel:

```
poetry install --with video,maite,notebook
```

(or `pip install 'databridge[video,maite,notebook]'` for the published package;
the `notebook` group provides the Jupyter kernel itself). Unlike the JSON-only
core, this demo exercises the video-touching paths end to end: section 0
renders real (tiny) mp4 clips with OpenCV, section 2 probes them for integrity,
the MOTChallenge writer in section 3 extracts their frames, and the MAITE
stream in section 4 decodes them with PyAV.

## 0. Build a tiny synthetic HMIE dataset

We write the on-disk HMIE layout that discovery walks: a full-length video
directory, snippet directories, a `scale/` annotation subfolder, and a
`seq_mp4/` video container. Labels, filenames, and ontology URIs are generic
placeholders. The clips are real one-second mp4s — a gray background with one
colored rectangle per annotated track — so every video-touching step in this
notebook (integrity probe, frame extraction, MAITE decoding) runs for real.

This toy tree mirrors the real layout but skips variants real datasets also use
(`seq_ts/` containers, multiple labeler subfolders, batch-level `scale/` dirs).
See the README section **"Dataset layout on disk"** for the full picture.

In [ ]:
import atexit
import json
import shutil
import tempfile
from pathlib import Path

import cv2
import numpy as np

VIDEO_FPS = 30.0
AFR = 5.0  # labeled keys per second: key k lands on video frame k * fps / afr
VIDEO_W, VIDEO_H = 320, 240
VIDEO_FRAMES = 30  # one second of video; the last labeled key (4) maps to frame 24


def make_annotation(tracks, *, task_id, afr=AFR, video_fps=VIDEO_FPS):
    """Minimal valid Scale Video Playback annotation dict."""
    annotations = {}
    for i, (label, bbox, n_frames) in enumerate(tracks):
        frames = [
            {
                "keyframeType": "start" if k == 0 else "middle",
                "isInferredKeyframe": False,
                "left": bbox[0],
                "top": bbox[1],
                "width": bbox[2],
                "height": bbox[3],
                "key": k,
                "attributes": {"is_occluded": "0%"},
                "timestamp_secs": k / afr,
            }
            for k in range(n_frames)
        ]
        annotations[f"track-uuid-{i:03d}"] = {
            "label": label,
            "geometry": "box",
            "frames": frames,
        }
    return {
        "task_id": task_id,
        "status": "completed",
        "type": "videoannotation",
        "params": {
            "annotation_frame_rate": afr,
            "videoMetadata": {"video": {"fps": video_fps}},
        },
        "response": {"annotations": annotations, "events": {}},
    }


def render_video(path, tracks, *, n_frames=VIDEO_FRAMES):
    """Render a real mp4: gray background, one colored rectangle per track."""
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(path), fourcc, VIDEO_FPS, (VIDEO_W, VIDEO_H))
    if not writer.isOpened():
        # cv2.VideoWriter fails *silently* (write() becomes a no-op) when the
        # build has no MPEG-4 encoder; fail here, not three sections later.
        raise RuntimeError(f"OpenCV could not open an mp4v VideoWriter for {path}")
    colors = [(60, 200, 60), (60, 60, 200), (200, 160, 60)]
    for _ in range(n_frames):
        frame = np.full((VIDEO_H, VIDEO_W, 3), 30, dtype=np.uint8)
        for i, (_label, (left, top, width, height), _n) in enumerate(tracks):
            corner = (int(left), int(top))
            opposite = (int(left + width), int(top + height))
            cv2.rectangle(frame, corner, opposite, colors[i % len(colors)], thickness=-1)
        writer.write(frame)
    writer.release()
    if not path.stat().st_size:
        raise RuntimeError(f"OpenCV wrote an empty clip to {path}")


def make_snippet(full_dir, name, tracks, *, task_id, src="SRC1", h="abc123"):
    """Create one snippet: scale/<CDAO ...>.json + seq_mp4/<name>.mp4."""
    snip = full_dir / name
    (snip / "scale").mkdir(parents=True, exist_ok=True)
    ann = snip / "scale" / f"CDAO_{src}_{name}.mp4_{h}.json"
    ann.write_text(json.dumps(make_annotation(tracks, task_id=task_id), indent=2))
    (snip / "seq_mp4").mkdir(exist_ok=True)
    render_video(snip / "seq_mp4" / f"{name}.mp4", tracks)


# Generic placeholder labels: two plain names + one ontology-style URI. The
# loader treats any string as a label and any http(s) URI as an ontology term.
BOAT = "boat"
VEHICLE = "vehicle"
WIDGET = "http://example.com/ontology/a/FOO_000"

root = Path(tempfile.mkdtemp(prefix="databridge_demo_"))
# Register cleanup now so the temp dir is removed when the kernel exits, even if
# a later cell errors or you stop before the explicit cleanup cell at the end.
atexit.register(shutil.rmtree, root, ignore_errors=True)

full = root / "clip_000000"
full.mkdir(parents=True)
(full / "clip_000000.json").write_text(json.dumps({"data_source": "synthetic"}))

make_snippet(
    full,
    "clip_000001",
    [
        (VEHICLE, (10.0, 20.0, 50.0, 40.0), 5),
        (BOAT, (80.0, 60.0, 30.0, 25.0), 5),
    ],
    task_id="task-0001",
)
make_snippet(
    full,
    "clip_000002",
    [
        (WIDGET, (12.0, 14.0, 44.0, 33.0), 4),
    ],
    task_id="task-0002",
)

# Show the tree we just built.
for p in sorted(root.rglob("*")):
    print(p.relative_to(root))

## 1. Load — `load_mot(root)` → `BoxTrackDataset`

One call walks the layout, parses each Scale annotation through the shared
schema, and returns the neutral `BoxTrackDataset` model. A dataset-wide category
map assigns each ontology label a stable integer id.

We pass `require_video=True` (needs the `video` extra) so each clip is probed
for its exact fps and frame count up front. Load once, reuse everywhere: this
same object feeds the export in section 3 and the MAITE view in section 4 —
on real collections that avoids re-walking the tree and re-probing every clip
per consumer.

In [ ]:
from databridge import load_mot

ds = load_mot(root, require_video=True)

print(f"sequences : {ds.sequence_count}")
print(f"boxes     : {ds.num_boxes}")
print(f"categories: {ds.categories}")

### Inspect the neutral model

A `BoxTrackDataset` holds `VideoSequence`s, each holding `BoxAnnotation`s. These
types live in `databridge.model` and know nothing about HMIE — they are the hub a
converter consumes. A few field semantics worth knowing before you rely on them:

- **`frame_index` is in video-frame space.** The loader maps each label's frame
  `key` (counted at the *annotation* frame rate) onto the video's real frame
  timeline: `frame# = key * fps / afr`. With `afr=5`, `fps=30` the labeled keys
  `0,1,2,…` land on video frames `0,6,12,…` — already aligned for frame-indexed
  formats like MOTChallenge, with no further math.
- **`num_frames` is an estimate unless the video was probed.** A default load
  never opens the video, so it reports a lower bound (max annotated frame + 1),
  not the true clip length. Because we passed `require_video=True` above, ours
  is the real probed count.
- **`category_id` is dataset-stable and 1-based**; an unlabeled track gets
  `category_id = -1` and no name. **`category_name`** is the last path segment of
  the label URI (`…/FOO_000` → `FOO_000`), or the label itself when it isn't a URI.

In [ ]:
seq = ds.sequences[0]
print("VideoSequence:")
print(f"  video_id   : {seq.video_id}")
print(f"  fps        : {seq.fps}")
print(f"  num_frames : {seq.num_frames}  (exact -- the load probed each clip)")
print(f"  status     : {seq.status}")
print(f"  boxes      : {len(seq.boxes)}")

box = seq.boxes[0]
print("\nBoxAnnotation (first box):")
print(f"  frame_index   : {box.frame_index}   # mapped to video-frame space")
print(f"  track_id      : {box.track_id}")
print(f"  category_id   : {box.category_id}  ({box.category_name})")
print(f"  bbox (l,t,w,h): {box.bbox}")
print(f"  is_inferred   : {box.is_inferred}")

## 2. Verify — `validate(root)`

Loading is best-effort and deliberately separate from validation. To find out
*why* data is bad, run the validator. Because section 0 rendered real clips, the
video-integrity probe stays on (the default). For JSON-only validation of large
collections pass `check_video_integrity=False` — the same fast path the CLI
exposes as `--skip-video-check`.

Note: the label histogram in the summary counts annotated **tracks**, not
per-frame boxes, so it won't match the box total from section 1.

In [ ]:
from databridge import validate

# workers=1 runs the validator serially -- it avoids spawning a process pool,
# which keeps the demo robust inside a notebook kernel. For large real datasets,
# drop it (or raise it) to fan validation across CPU cores.
result = validate(root, workers=1)
print(result.summary(use_color=False))

## 3. Export — `write(ds, dest, ...)`

`write()` hands the neutral `BoxTrackDataset` to the registered writer for
`output_format`. By default it writes for side effects and returns `None`; pass
`verbose=True` (as we do below) to get the list of files written back — the
per-frame file list can be long, so it is opt-in to keep interactive output
quiet. Because writers bind to the
neutral model — never to an input format — any registered **box-track/MOT**
input can target any registered **box-track/MOT** output.
`available_formats()` lists every loader in the package (including task
siblings such as Hugging Face video classification), while
`available_output_formats()` lists the current `BoxTrackDataset` writers. For
the disk-to-disk case there is also `convert(src, dest, input_format=...,
output_format=..., read_options=...)`, which is exactly load + write in one
call. Below we use both convenience paths: `write()` because we already hold
the loaded (and video-probed) dataset from section 1, then an executed
`convert()` call to show the one-shot disk-to-disk API.

Here we export the synthetic HMIE dataset as a **MOTChallenge** benchmark root.
The neutral model already carries MOT's ground-truth `gt.txt` columns in the
right units:

| MOTChallenge `gt.txt` column | source on `BoxAnnotation` |
|------------------------------|---------------------------|
| `frame`                      | `frame_index` (already video-frame space) |
| `id`                         | `track_id` (unique *within a sequence*) |
| `bb_left, bb_top, bb_width, bb_height` | `bbox` (pixels: left, top, w, h) |
| `conf`                       | `1` for ground truth |
| `class`                      | mapped from `category_id` |

The format-specific decisions are the writer's:

- **One `gt.txt` per sequence** — `track_id` is unique only within a sequence,
  which is exactly MOT's scope.
- **Frame images** — MOTChallenge ships frames as `img1/000001.jpg…`, so the
  writer extracts them from the source video (OpenCV). This is why the demo
  renders real clips.
- **Best-effort mapping** — boxes the target format cannot represent are
  dropped with a warning, not an exception; destination/IO failures do raise.

The `require_video=True` load in section 1 already probed each clip's exact
frame count, so the extracted images line up with `frame_index` one-to-one.

In [ ]:
from databridge import available_formats, available_output_formats, convert, write

print("registered input loaders    :", [fmt.value for fmt in available_formats()])
print("BoxTrack output writers     :", [fmt.value for fmt in available_output_formats()])
print("note: Hugging Face video classification is a loader, but not a BoxTrack/MOT conversion input")

mot_root = Path(tempfile.mkdtemp(prefix="databridge_demo_mot_"))
atexit.register(shutil.rmtree, mot_root, ignore_errors=True)

# In-memory convenience: reuse the object we loaded once in section 1.
# write()/convert() return None by default (writing is a side effect); pass
# verbose=True to get the list of written files back for inspection here.
written = write(ds, mot_root, output_format="motchallenge", verbose=True)

images = [p for p in written if p.suffix == ".jpg"]
print()
print(f"write(ds, ...): wrote {len(written)} files ({len(images)} frame images); the rest:")
for path in written:
    if path.suffix != ".jpg":
        print(f"  {path.relative_to(mot_root)}")

# Disk-to-disk convenience: load + write in one call. This re-walks/re-probes
# the tiny synthetic tree, so prefer write(ds, ...) when you already hold a
# loaded dataset, but convert(...) is the shortest API for batch conversion.
mot_convert_root = Path(tempfile.mkdtemp(prefix="databridge_demo_mot_convert_"))
atexit.register(shutil.rmtree, mot_convert_root, ignore_errors=True)
converted = convert(
    root,
    mot_convert_root,
    input_format="hmie",
    output_format="motchallenge",
    read_options={"require_video": True},
    verbose=True,
)
converted_images = [p for p in converted if p.suffix == ".jpg"]
print()
print(f"convert(root, ...): wrote {len(converted)} files ({len(converted_images)} frame images)")

In [ ]:
# The written tree is a standard MOTChallenge benchmark root: per-sequence
# gt/gt.txt (frame, id, bb_left, bb_top, bb_width, bb_height, conf, class,
# visibility) plus seqinfo.ini describing the extracted image stream.
seq_dir = sorted((mot_root / "train").iterdir())[0]
print(f"--- {seq_dir.name}/gt/gt.txt ---")
print((seq_dir / "gt" / "gt.txt").read_text())
print(f"--- {seq_dir.name}/seqinfo.ini ---")
print((seq_dir / "seqinfo.ini").read_text())

## 4. Consume — the loaded dataset *is* a MAITE MOT dataset

MAITE interoperability is not an adapter or a conversion call: `BoxTrackDataset`
implements MAITE's multi-object-tracking dataset protocol directly. MAITE
protocols are structural (conformance is by shape), so `databridge` never
imports `maite` at runtime — but with `maite` installed we can assert
conformance explicitly, and MAITE workflows such as `maite.tasks.predict`
accept the dataset as-is. The section-1 load already probed each clip
(`require_video=True`), which is what gives the MOT view exact frame counts.

The object wears two hats, with two distinct "size" views:

- **MAITE item view** — `len(ds)` / `ds[i]` / iterating `ds`: one item per
  *video-bearing* sequence (MOT needs pixels). `ds[i]` yields
  `(VideoStream, MultiobjectTrackingTarget, DatumMetadata)`.
- **Record view** — `ds.sequence_count` / `ds.iter_sequences()` /
  `ds.sequences`: every loaded sequence, video-less ones included. This is what
  the validator and the converters walk.

The `VideoStream` is lazy: frames decode (via PyAV) only as you iterate it.
Under the default `empty_frame_policy="annotated"` the stream yields exactly
the labeled frames, aligned one-to-one with `target.frame_tracks`.

**Careful: two `frame_index` spaces.** `DecodedFrame.frame_index` follows
MAITE's definition — the zero-based position within the *yielded* stream —
which is *not* the video-frame space `BoxAnnotation.frame_index` uses
(section 1). Under the `"annotated"` policy our stream positions `0,1,2,…`
correspond to video frames `0,6,12,…`. To pair a decoded frame with
annotations, use its stream position to index `target.frame_tracks` (they are
aligned one-to-one), or recover the video frame from `time_s * fps` — never
match `DecodedFrame.frame_index` against `BoxAnnotation.frame_index` directly.

MOT-view options (`empty_frame_policy`, decoder, `dataset_id`) are configured
with `ds.with_mot_options(...)`, which returns a configured copy.

In [ ]:
import itertools

from maite.protocols import multiobject_tracking as mot

print(f"isinstance(ds, maite MOT Dataset): {isinstance(ds, mot.Dataset)}")
print(f"MAITE items (video-bearing)      : {len(ds)}")
print(f"record view (all sequences)      : {ds.sequence_count}")
print(f"dataset metadata                 : {ds.metadata}")

stream, target, datum_metadata = ds[0]
print(f"\ndatum metadata: {datum_metadata}")
print(f"labeled frames in target: {len(target.frame_tracks)}")

first = target.frame_tracks[0]
print("\nframe 0 target (one row per box):")
print(f"  boxes (xyxy): {first.boxes.tolist()}")
print(f"  labels      : {first.labels.tolist()}")
print(f"  track_ids   : {first.track_ids.tolist()}")

print("\ndecoding the first two frames of the lazy stream:")
fps = ds.sequences[0].fps
for frame in itertools.islice(stream, 2):
    # frame.frame_index is the position in the *yielded* stream; the video
    # frame it came from is recovered from the timestamp.
    video_frame = round(frame.time_s * fps)
    print(
        f"  stream position {frame.frame_index} (video frame {video_frame})  "
        f"pixels={frame.pixels.shape} uint8  time_s={frame.time_s:.3f}"
    )

## Where to go next

- **Batch validation with HTML reports** — `validators/hmie.ipynb` runs the
  validator across a whole collection of datasets and renders roll-up reports.
- **Other box-track/MOT input formats** — the same
  `load_mot(root, dataset_format=...)` call loads MOTChallenge, TAO, VisDrone
  video, and flat-MP4 trees.
- **Video classification** — the Hugging Face video-classification loader is
  registered too, but it returns a `VideoClassificationDataset` via `load(...)`
  or `load_huggingface_video_classification(...)`; it does not use the
  `BoxTrackDataset` writers or MAITE MOT view.
- **Other output formats** — swap `output_format` to `"tao"`,
  `"visdrone_video"`, or `"hmie"`; the same neutral `BoxTrackDataset` feeds
  every MOT writer.


In [ ]:
# Optional explicit cleanup. All trees are also registered with atexit, so
# they are removed when the kernel exits even if you skip this cell.
for tree in (root, mot_root, mot_convert_root):
    shutil.rmtree(tree, ignore_errors=True)
    print(f"removed {tree}")